In [50]:
import json

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------

with open("experimental_outputs/label_change_intervention_results.json", "r") as f:
    data = json.load(f)

VALID_SUBSETS = {"true", "false"}
VALID_STRENGTHS = {-2, -1, 0, 1, 2}
VALID_LAYERS = {9, 10, 11, 12, 13}

# --------------------------------------------------
# FILTER + BUILD DICTIONARY
# --------------------------------------------------

result = {}

for d in data:
    # filter subset
    if d.get("subset") not in VALID_SUBSETS:
        continue

    # filter intervention strength
    strength = d.get("intervention_strength")
    if strength not in VALID_STRENGTHS:
        continue

    # filter hidden state layers
    layers = {hs[0] for hs in d.get("hidden_states", [])}
    if not layers.issubset(VALID_LAYERS):
        continue

    model = d["model"]
    probe = d["probe class"]
    subset = d["subset"]
    p_diff = d["p_diff"]

    # train_datasets is a list → make it hashable & order-invariant
    train_dataset = "+".join(sorted(d["train_datasets"]))

    key = (model, probe, train_dataset, subset, strength)
    result[key] = p_diff

# --------------------------------------------------
# RESULT
# --------------------------------------------------

print(f"Number of entries: {len(result)}")


Number of entries: 128


In [54]:
model = 'llama-3.2-3B-Instruct'
probe = 'MMProbe'
train_dataset = 'random'

strength = 2
NIE_false_to_true = (result[(model, probe, train_dataset, 'false', strength)] - result[(model, probe, train_dataset, 'false', 0)]) / (result[(model, probe, train_dataset, 'true', 0)] - result[(model, probe, train_dataset, 'false', 0)])
NIE_true_to_false = (result[(model, probe, train_dataset, 'true', -strength)] - result[(model, probe, train_dataset, 'true', 0)]) / (result[(model, probe, train_dataset, 'false', 0)] - result[(model, probe, train_dataset, 'true', 0)])
print(NIE_false_to_true, NIE_true_to_false)

0.0 -0.004338394793926247
